# Prairie Vigie – notebook 3 : consolidation

| Bloc | Question | Règle écrite avant les résultats |
|---|---|---|
| A1 | Nos « conversions » sont-elles de vraies mises en culture durables ? | ≥ 75 % encore cultivées 2 ans après |
| A2 | Concordent-elles avec ce que déclarent les agriculteurs (champ `retournmt_` du RPG 2024) ? | exploratoire (signification du champ à confirmer avec la DRAAF) |
| A3 | Que deviennent les prairies « sorties du RPG » ? | descriptif (conversions différées) |
| B | Le modèle bat-il des approches plus simples ? Sur quels seuils ? | on retient le modèle **le plus simple** dont le lift top 10 % est à moins de 0,2 du meilleur |
| C | Les probabilités sont-elles fiables (calibration) ? | descriptif : courbe par décile + score de Brier |
| D | Le régime d'interdiction de retournement, annoncé fin octobre 2023 puis levé rétroactivement au 1er janvier 2024 (arrêté du 25 novembre 2024) a-t-il coïncidé avec moins de conversions ? | observé vs attendu par le modèle (pas une preuve causale) |
| E | Combien d'hectares risquent d'être convertis par interco lors de la campagne 2027, avec quelle incertitude ? (commune et maille de 1 km : fichiers `_INTERNE`, prédiction non validée à ces échelles) | intervalles validés par commune sur 2025 (cellule C) ; secret : rien sous 5 prairies |
| F | Pourquoi un secteur est-il à risque ? | importance + courbes de dépendance |
| G | Quel coût énergétique ? | mesuré avec CodeCarbon (sobriété numérique) |

In [ ]:
# 0. Sur Google Colab, le dossier DOSSIER est créé (ou repris) dans « Mon Drive » .
#    Les 4 notebooks doivent utiliser le même dossier.
DOSSIER = "datathon_prairies"
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")                                   # point de montage standard de Colab
    os.makedirs(os.path.join("/content/drive/MyDrive", DOSSIER), exist_ok=True)
    os.chdir(os.path.join("/content/drive/MyDrive", DOSSIER))
except ImportError:
    pass                                                            # hors Colab : dossier courant
print("Dossier de travail :", os.getcwd())
if not os.path.exists("sorties_prairies"):
    print("⚠ Dossier vide : lancer d'abord les notebooks précédents dans ce même dossier.")

In [ ]:
# 0 bis. Bibliothèques
!pip -q install geopandas pyogrio pyarrow scikit-learn codecarbon matplotlib

In [ ]:
# 1. Paramètres
import time, gc
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, requests
from shapely.geometry import shape, box
from shapely.validation import make_valid
from shapely.ops import unary_union

DEPS = ["14", "27", "50", "61", "76"]
WFS = "https://data.geopf.fr/wfs/ows"
PAGE, DALLE = 5000, 20_000
CACHE = Path("cache_rpg"); CACHE.mkdir(exist_ok=True)
SORTIES = Path("sorties_prairies"); SORTIES.mkdir(exist_ok=True)
SESSION = requests.Session()
MIN_PP_MAILLE = 5                            # aucun résultat calculé pour moins de 5 prairies

PP, HERBE, GEL = 18, {17, 18, 19}, {11, 12, 13}
ARABLE = {1, 2, 3, 4, 5, 6, 7, 8, 9, 14, 15, 16, 24, 25, 26}
def classe(g):
    g = int(g) if pd.notna(g) else -1
    return ("pp" if g == PP else "herbe" if g in HERBE else "arable" if g in ARABLE
            else "gel" if g in GEL else "autre")

# Mesure de la consommation d'énergie du notebook (sobriété numérique)
try:
    from codecarbon import EmissionsTracker
    TRACKER = EmissionsTracker(project_name="sentinelle_v5", log_level="error", save_to_file=False)
    TRACKER.start()
except Exception as e:
    TRACKER = None
    print("Mesure énergétique indisponible :", e)
T_DEBUT = time.time()
FIG = SORTIES / "figures"; FIG.mkdir(exist_ok=True)
# Versions des bibliothèques (reprises dans requirements.txt du dépôt)
import sys, importlib.metadata as md
VERSIONS = {}
for p in ["numpy", "pandas", "geopandas", "shapely", "pyogrio", "pyarrow", "scikit-learn", "matplotlib", "requests", "codecarbon"]:
    try:
        VERSIONS[p] = md.version(p)
    except md.PackageNotFoundError:
        print("Bibliothèque absente :", p)
(SORTIES / "versions_bibliotheques.txt").write_text(f"# python {sys.version.split()[0]}\n" +
    "\n".join(f"{k}=={v}" for k, v in VERSIONS.items()) + "\n", encoding="utf-8")
print("Python", sys.version.split()[0], VERSIONS)

In [ ]:
# 1. Outils communs : WFS par dalles adaptatives, lecture du cache, jointure par point
def get(params):
    for essai in range(6):
        try:
            r = SESSION.get(WFS, params=params, timeout=300)
            if r.status_code == 200:
                return r.json()
        except (requests.RequestException, ValueError):
            pass
        time.sleep(5 * 2 ** essai)
    raise RuntimeError(f"WFS indisponible : {params.get('TYPENAMES')}")

def dalle(couche, champs, x0, y0, x1, y1, zone, sortie, vus):
    if not box(x0, y0, x1, y1).intersects(zone):
        return
    page = get({"SERVICE": "WFS", "VERSION": "2.0.0", "REQUEST": "GetFeature", "TYPENAMES": couche,
                "SRSNAME": "EPSG:2154", "BBOX": f"{x0},{y0},{x1},{y1},EPSG:2154",
                "OUTPUTFORMAT": "application/json", "COUNT": PAGE})
    feats = page.get("features", [])
    m = page.get("numberMatched", page.get("totalFeatures"))
    if len(feats) >= PAGE or (isinstance(m, int) and m > len(feats)):
        xm, ym = (x0 + x1) / 2, (y0 + y1) / 2
        for a, b, c, d in [(x0, y0, xm, ym), (xm, y0, x1, ym), (x0, ym, xm, y1), (xm, ym, x1, y1)]:
            dalle(couche, champs, a, b, c, d, zone, sortie, vus)
        return
    for f in feats:
        if f.get("id") in vus or not f.get("geometry"):
            continue
        vus.add(f.get("id"))
        props = {k.lower(): v for k, v in (f.get("properties") or {}).items()}
        sortie.append({**{k: props.get(k) for k in champs}, "geometry": shape(f["geometry"])})   # liste blanche

COMMUNES = gpd.GeoDataFrame(pd.concat([gpd.read_parquet(CACHE / f"communes_{d}.parquet").assign(dep=d) for d in DEPS],
                                      ignore_index=True), geometry="geometry", crs=2154)
CONTOURS = {d: make_valid(unary_union(COMMUNES.loc[COMMUNES.dep == d, "geometry"].values)) for d in DEPS}

def rpg(dep, annee):
    g = gpd.read_parquet(CACHE / f"rpg_{dep}_{annee}_v3.parquet")
    g["pid"] = np.arange(len(g))
    g["classe"] = g["code_group"].apply(classe)
    return g

def points(geoms):
    return gpd.GeoDataFrame(geometry=gpd.GeoSeries(geoms, crs=2154).representative_point().values, crs=2154)

def au_point(pts, g, cols):
    j = gpd.sjoin(pts, g[cols + ["geometry"]], how="left", predicate="within")
    return j[~j.index.duplicated()].reindex(pts.index)[cols]

# Libellés des codes de culture (table officielle RPG 2024)
try:
    LIB = pd.DataFrame([f["properties"] for f in get({"SERVICE": "WFS", "VERSION": "2.0.0", "REQUEST": "GetFeature",
                        "TYPENAMES": "RPG.2024:codes_cultures", "OUTPUTFORMAT": "application/json", "COUNT": 5000})["features"]])
    LIB = dict(zip(LIB["code"], LIB["libelle"]))
except Exception as e:
    LIB = {}
    print("Libellés indisponibles :", e)
print(f"{len(LIB)} libellés de cultures")

# Tables de variables issues des notebooks 1 et 2 (une ligne par prairie et par année de départ a0)
TAB = pd.concat([pd.read_parquet(SORTIES / f"variables_{d}.parquet") for d in DEPS], ignore_index=True)
NOUV = pd.concat([pd.read_parquet(SORTIES / f"v4_{d}.parquet") for d in DEPS], ignore_index=True)
ALL = pd.concat([TAB[TAB.a0 <= 2021], NOUV], ignore_index=True)     # a0 = 2019 … 2023
print(ALL.groupby("a0")[["y", "y2"]].agg(["count", "sum"]).to_string())

In [ ]:
# A1 + A3. Qualité des étiquettes : persistance à 2 ans, cultures de remplacement, conversions différées
lignes, cultures = [], []
for dep in DEPS:
    R = {}
    for a0 in range(2019, 2024):
        for a in (a0, a0 + 1, a0 + 2):                  # fenêtre glissante : 3 années en mémoire au maximum
            if a <= 2025 and a not in R:
                R[a] = rpg(dep, a)
        for a in [a for a in R if a < a0]:
            del R[a]
        gc.collect()
        t = ALL[(ALL.dep == dep) & (ALL.a0 == a0)]
        g0 = R[a0].set_index("pid")
        conv = t[t.y == 1]
        pts = points(g0.loc[conv.pid, "geometry"].values)
        cult1 = au_point(pts, R[a0 + 1], ["code_cultu", "classe"])
        cultures.append(cult1.assign(dep=dep, a0=a0))
        ligne = {"dep": dep, "transition": f"{a0}->{a0 + 1}", "conversions": len(conv)}
        if a0 + 2 in R:
            c2 = au_point(pts, R[a0 + 2], ["classe"])["classe"]
            ligne["encore_cultivees_N+2"] = round(c2.isin(["arable", "gel"]).mean(), 3)
            ligne["revenues_en_herbe_N+2"] = round(c2.isin(["pp", "herbe"]).mean(), 3)
            sortie = t[t.devenir == "absent"]
            ps = points(g0.loc[sortie.pid, "geometry"].values)
            cs = au_point(ps, R[a0 + 2], ["classe"])["classe"]
            ligne["sorties_rpg"] = len(sortie)
            ligne["sorties_puis_cultivees_N+2"] = int(cs.isin(["arable", "gel"]).sum())      # conversions différées
            ligne["sorties_puis_reapparues_en_herbe"] = int(cs.isin(["pp", "herbe"]).sum())
        lignes.append(ligne)
    del R; gc.collect()
QUAL = pd.DataFrame(lignes)
QUAL.to_csv(SORTIES / "qualite_etiquettes.csv", index=False)
print(QUAL.to_string(index=False))
pers = (QUAL["encore_cultivees_N+2"] * QUAL["conversions"]).sum() / QUAL.loc[QUAL["encore_cultivees_N+2"].notna(), "conversions"].sum()
CRIT_PERSIST = bool(pers >= 0.75)
print(f"\nPersistance moyenne à 2 ans : {pers:.1%} -> {'OK' if CRIT_PERSIST else 'ÉTIQUETTES BRUITÉES'} (seuil 75 %)")

CU = pd.concat(cultures)
CU["libelle"] = CU["code_cultu"].map(LIB).fillna(CU["code_cultu"])
rep = CU.groupby("libelle").size().sort_values(ascending=False)
print("\nCultures qui remplacent les prairies converties (2019-2024, Normandie) :")
print((rep / rep.sum()).head(10).round(3).to_string())
rep.to_csv(SORTIES / "cultures_de_remplacement.csv")

In [ ]:
# A2. Concordance avec le champ déclaratif « retournmt_ » du RPG 2024 (exploratoire)
COUCHE24 = "IGNF_RPG_PARCELLES-AGRICOLES-CATEGORISEES_2024:parcelles_agricole_categorisees_2024"
CHAMPS24 = ["retournmt_", "reconver_p", "labour", "cat_cult_p"]          # AUCUN identifiant conservé
def oui(v):
    return str(v).strip().upper() in {"1", "TRUE", "OUI", "O", "T", "Y", "VRAI"}
res = []
for dep in DEPS:
    f = CACHE / f"rpg24cat_{dep}.parquet"
    if f.exists():
        g24 = gpd.read_parquet(f)
    else:
        lignes, vus = [], set()
        xmin, ymin, xmax, ymax = CONTOURS[dep].bounds
        for x in np.arange(xmin, xmax, DALLE):
            for y in np.arange(ymin, ymax, DALLE):
                dalle(COUCHE24, CHAMPS24, x, y, x + DALLE, y + DALLE, CONTOURS[dep], lignes, vus)
        g24 = gpd.GeoDataFrame(lignes, geometry="geometry", crs=2154)
        g24 = g24[g24.representative_point().within(CONTOURS[dep])]
        g24.to_parquet(f)
    if dep == DEPS[0]:
        for c in CHAMPS24:
            print(f"Valeurs de {c} :", g24[c].astype(str).value_counts().head(6).to_dict())
    g24["flag_retournement"] = g24["retournmt_"].apply(oui).astype(int)
    # nos conversions 2023->2024 : la parcelle 2024 sous le point porte-t-elle le drapeau ?
    t = NOUV[(NOUV.dep == dep) & (NOUV.a0 == 2023)]
    g23 = rpg(dep, 2023).set_index("pid")
    pp23 = t[["pid", "y"]]
    flag = au_point(points(g23.loc[pp23.pid, "geometry"].values), g24, ["flag_retournement"])["flag_retournement"]
    pp23 = pp23.assign(flag=flag.fillna(0).astype(int).values)
    res.append({"dep": dep, "nos_conversions": int(pp23.y.sum()),
                "dont_drapeau_declare": int(((pp23.y == 1) & (pp23.flag == 1)).sum()),
                "drapeaux_sur_PP_2023": int(pp23.flag.sum()),
                "dont_detectes_par_nous": int(((pp23.y == 1) & (pp23.flag == 1)).sum())})
A2 = pd.DataFrame(res)
tot = A2.sum(numeric_only=True)
print(A2.to_string(index=False))
if tot["drapeaux_sur_PP_2023"] > 0:
    print(f"\nPart de nos conversions portant le drapeau : {tot.dont_drapeau_declare / max(tot.nos_conversions, 1):.1%} ; "
          f"part des drapeaux que nous détectons : {tot.dont_detectes_par_nous / tot.drapeaux_sur_PP_2023:.1%}")
else:
    print("\nAucun drapeau exploitable : champ vide ou de sens différent -> question à poser à la DRAAF.")
A2.to_csv(SORTIES / "concordance_retournement_declare_2024.csv", index=False)

In [ ]:
# B. Évaluation complète : modèles, références, seuils, départements, prairies anciennes
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.isotonic import IsotonicRegression

CODES = list(ALL["code_cultu"].value_counts().index[:8])
def onehot(df):
    for c in CODES:
        df[f"code_{c}"] = (df["code_cultu"] == c).astype(int)
    return df
ALL = onehot(ALL)
NON_VARS = {"pid", "y", "y2", "devenir", "transition", "code_cultu", "dep", "a0", "cx1", "cy1", "commune"}
VARS = [c for c in ALL.columns if c not in NON_VARS]
MODELES = {
    "gradient boosting": lambda: HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, random_state=0),
    "régression logistique": lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                                                  LogisticRegression(max_iter=3000, C=1.0)),
}
def fit(nom, tr, cible):
    cols = [v for v in VARS if tr[v].notna().any()]
    return MODELES[nom]().fit(tr[cols], tr[cible]), cols

def metriques(y, s):
    y, s = np.asarray(y, float), np.nan_to_num(np.asarray(s, float))
    o = {"AUC_PR": round(average_precision_score(y, s), 4)}
    for k in (5, 10, 20):
        top = np.argsort(-s)[:max(1, int(k / 100 * len(y)))]
        o[f"lift_{k}"] = round(y[top].mean() / y.mean(), 2)
        o[f"rappel_{k}"] = round(y[top].sum() / y.sum(), 3)
    return o

# On calcule d'abord la zone « Normandie » (qui remplit les scores), puis les départements
def campagne_ordonnee(cible, a0_test, retard):
    d0 = DEPS
    DEPS_ORD = ["Normandie"] + d0
    base = ALL[ALL[cible].notna()]
    te = base[base.a0 == a0_test]
    scores = {}
    for nom in MODELES:
        s = pd.Series(np.nan, index=te.index)
        for d in d0:
            tr = base[(base.dep != d) & (base.a0 < a0_test - retard)]
            m, cols = fit(nom, tr, cible)
            idx = te.index[te.dep == d]
            s.loc[idx] = m.predict_proba(te.loc[idx, cols])[:, 1]
        scores[nom] = s
    out = []
    for zone in DEPS_ORD:
        t = te if zone == "Normandie" else te[te.dep == zone]
        for nom in MODELES:
            s = scores[nom].loc[t.index]
            out.append({"cible": cible, "zone": zone, "modele": nom, **metriques(t[cible], s)})
            anc = (t.ancienne == 1).values
            out.append({"cible": cible, "zone": f"{zone} (anciennes)", "modele": nom, **metriques(t[cible].values[anc], s.values[anc])})
        for nom, col in [("réf. part arable 1 km", "part_arable_1km"), ("réf. conversions voisines", "taux_ret_prec_1km")]:
            out.append({"cible": cible, "zone": zone, "modele": nom, **metriques(t[cible], t[col])})
    return pd.DataFrame(out), scores, te

pd.set_option("display.width", 250)
EV1, S1, TE1 = campagne_ordonnee("y", 2022, 0)       # 1 an : 2022 -> conversions 2022-2023
EV2, S2, TE2 = campagne_ordonnee("y2", 2023, 1)      # 2 ans, comme dans le notebook 2 : entraînement a0 ≤ 2021, données 2023 -> conversions 2024-2025
EV = pd.concat([EV1.assign(test="1 an, 2022->23"), EV2.assign(test="2 ans, 2023 -> 2024->25")])
EV.to_csv(SORTIES / "evaluation_complete.csv", index=False)
print(EV[EV.zone.isin(["Normandie", "Normandie (anciennes)"])].to_string(index=False))
print("\nPar département (lift top 10 %, 2 ans) :")
print(EV2[~EV2.zone.str.contains("ancien")].pivot(index="zone", columns="modele", values="lift_10").to_string())

# Choix du modèle (règle écrite) : le plus simple à moins de 0,2 du meilleur lift top 10 % (2 ans, Normandie)
l = EV2[EV2.zone == "Normandie"].set_index("modele")["lift_10"]
MODELE_RETENU = "régression logistique" if l["régression logistique"] >= l["gradient boosting"] - 0.2 else "gradient boosting"
print(f"\nModèle retenu : {MODELE_RETENU} (lift 2 ans : logistique {l['régression logistique']}, boosting {l['gradient boosting']})")

# Concordance par commune EN TAUX (retire l'effet taille) – 2 ans
te = TE2.assign(s=S2[MODELE_RETENU])
com = te.groupby(["dep", "commune"]).agg(ha=("surface_ha", "sum"), risque=("s", "mean"),
        ref=("part_arable_1km", "mean"), ha_conv=("surface_ha", lambda x: x[te.loc[x.index, "y2"] == 1].sum()))
com = com[com.ha >= 20]
com["taux"] = com.ha_conv / com.ha
print(f"Communes (≥ 20 ha de prairies) : {len(com)} ; Spearman risque/taux = {com.risque.corr(com.taux, method='spearman'):.3f} ;"
      f" référence part arable/taux = {com.ref.corr(com.taux, method='spearman'):.3f}")

In [ ]:
# C. Calibration : les probabilités annoncées correspondent-elles aux fréquences observées ?
import matplotlib.pyplot as plt
base = ALL[ALL["y2"].notna()]
m, cols = fit(MODELE_RETENU, base[base.a0 <= 2020], "y2")             # entraînement : 2019-2020
cal = IsotonicRegression(out_of_bounds="clip").fit(m.predict_proba(base.loc[base.a0 == 2021, cols])[:, 1],
                                                     base.loc[base.a0 == 2021, "y2"])   # calibration : 2021
te = base[base.a0 == 2023]                                                # test : conversions 2024-2025
brut = m.predict_proba(te[cols])[:, 1]
p = cal.predict(brut)
dec = pd.qcut(pd.Series(p).rank(method="first"), 10, labels=False)
tab = pd.DataFrame({"decile": dec, "prevu": p, "observe": te["y2"].values}).groupby("decile").mean()
print(tab.round(4).to_string())
print(f"Brier brut {brier_score_loss(te.y2, brut):.5f} ; calibré {brier_score_loss(te.y2, p):.5f} ; "
      f"taux réel {te.y2.mean():.4f} ; moyenne prévue (calibrée) {p.mean():.4f}")
plt.figure(figsize=(4.5, 4.5)); plt.plot(tab.prevu, tab.observe, "o-"); m_ = max(tab.max()); plt.plot([0, m_], [0, m_], "--", c="grey")
plt.xlabel("probabilité prévue"); plt.ylabel("fréquence observée (2025)"); plt.title("Calibration, données jamais vues")
plt.tight_layout(); plt.savefig(FIG / "calibration_2025.png", dpi=150); plt.show()

# Validation à la commune (2025, données jamais vues) : sert à mesurer l'incertitude réelle des intervalles
vc = te.assign(p=p).groupby(["dep", "commune"]).apply(lambda d: pd.Series({
        "n": len(d), "ha_pp": d.surface_ha.sum(), "attendu": (d.p * d.surface_ha).sum(),
        "var": (d.p * (1 - d.p) * d.surface_ha ** 2).sum(),
        "observe": d.loc[d.y2 == 1, "surface_ha"].sum()})).reset_index()
vc = vc[vc.n >= MIN_PP_MAILLE]
# Les intervalles « binomiaux » ignorent l'incertitude du modèle : on mesure l'écart réel (surdispersion)
PHI = float(max(1.0, ((vc.observe - vc.attendu) ** 2).sum() / vc["var"].sum()))
for nom, f in [("binomial seul", 1.0), (f"corrigé (×√{PHI:.1f})", np.sqrt(PHI))]:
    sd = np.sqrt(vc["var"]) * f
    cov = ((vc.observe >= vc.attendu - 1.645 * sd) & (vc.observe <= vc.attendu + 1.645 * sd)).mean()
    print(f"Couverture de l'intervalle à 90 % par commune, {nom} : {cov:.0%}")
r = vc[vc.ha_pp >= 20]
print(f"Communes (≥ 20 ha) : Spearman taux attendu / taux observé = "
      f"{(r.attendu / r.ha_pp).corr(r.observe / r.ha_pp, method='spearman'):.3f} ; "
      f"total attendu {vc.attendu.sum():,.0f} ha / observé {vc.observe.sum():,.0f} ha")
vc.to_csv(SORTIES / "validation_communes_2025_INTERNE.csv", index=False)   # usage interne : prédiction à la commune

In [ ]:
# D. Campagne 2023→24 et régime d'interdiction de retournement, annoncé fin octobre 2023 puis levé rétroactivement au 1er janvier 2024 (arrêté du 25 novembre 2024) :
#    conversions présumées observées vs attendues (modèle 1 an entraîné AVANT le régime)
base = ALL[ALL["y"].notna()]
m, cols = fit(MODELE_RETENU, base[base.a0 <= 2020], "y")
cal1 = IsotonicRegression(out_of_bounds="clip").fit(m.predict_proba(base.loc[base.a0 == 2021, cols])[:, 1],
                                                      base.loc[base.a0 == 2021, "y"])
out = []
for a0, nom in [(2022, "2022->23 (témoin, avant le régime)"), (2023, "2023->24 (année du régime)")]:
    t = base[base.a0 == a0]
    p = cal1.predict(m.predict_proba(t[cols])[:, 1])
    att, sd = (p * t.surface_ha).sum(), np.sqrt((p * (1 - p) * t.surface_ha ** 2).sum())
    obs = t.loc[t.y == 1, "surface_ha"].sum()
    out.append({"periode": nom, "ha_observes": round(obs), "ha_attendus": round(att),
                "intervalle_90": f"{att - 1.645 * sd:,.0f} – {att + 1.645 * sd:,.0f}", "observe/attendu": round(obs / att, 2)})
REG = pd.DataFrame(out)
print(REG.to_string(index=False))
print("Lecture : si le témoin est proche de 1 et l'année du régime nettement en dessous, la baisse n'est pas expliquée",
      "par le contexte agricole décrit par le modèle. C'est une observation, pas une preuve de cause à effet.")
REG.to_csv(SORTIES / "effet_regime_2023_2024.csv", index=False)

In [ ]:
# E. Modèle final : hectares attendus convertis lors de la campagne 2027 (prairies 2025 encore en herbe en 2026),
#    par interco (diffusable), par commune et maille de 1 km (fichiers _INTERNE : prédiction non validée à ces échelles),
#    intervalle à 90 % corrigé de la surdispersion mesurée en C, rien sous 5 prairies
base = ALL[ALL["y2"].notna()]
m, cols = fit(MODELE_RETENU, base[base.a0 <= 2022], "y2")             # toutes les étiquettes sauf la dernière année
calf = IsotonicRegression(out_of_bounds="clip").fit(m.predict_proba(base.loc[base.a0 == 2023, cols])[:, 1],
                                                      base.loc[base.a0 == 2023, "y2"])     # calibrée sur 2024-2025
S = onehot(pd.concat([pd.read_parquet(SORTIES / f"score_{d}_2025.parquet") for d in DEPS], ignore_index=True))
for v in VARS:
    if v not in S:
        S[v] = np.nan
S["p"] = calf.predict(m.predict_proba(S[cols])[:, 1])
S["top10"] = (S.p >= S.p.quantile(0.9)).astype(int)
S[["dep", "pid", "commune", "cx1", "cy1", "surface_ha", "ancienne", "p", "top10"]].to_parquet(SORTIES / "risque_2027_calibre_PRIVE.parquet")
info = COMMUNES[["code", "nom", "codeEpci", "dep"]].rename(columns={"code": "commune"})
S = S.merge(info, on=["dep", "commune"], how="left")
def bilan(df, cles):
    g = df.assign(e=df.p * df.surface_ha, v=df.p * (1 - df.p) * df.surface_ha ** 2,
                  t=df.top10 * df.surface_ha).groupby(cles, dropna=False)
    b = g.agg(n_pp=("pid", "size"), ha_pp=("surface_ha", "sum"), ha_attendus_campagne_2027=("e", "sum"), var=("v", "sum"),
              ha_secteurs_top10=("t", "sum")).reset_index()
    sd = np.sqrt(b.pop("var") * PHI)                 # PHI mesuré sur 2025 (cellule C)
    b["ha_bas_90"] = (b.ha_attendus_campagne_2027 - 1.645 * sd).clip(lower=0)
    b["ha_haut_90"] = b.ha_attendus_campagne_2027 + 1.645 * sd
    b["taux_attendu"] = b.ha_attendus_campagne_2027 / b.ha_pp
    secret = b.n_pp < MIN_PP_MAILLE
    b.loc[secret, ["ha_attendus_campagne_2027", "ha_bas_90", "ha_haut_90", "taux_attendu", "ha_secteurs_top10"]] = np.nan
    return b.round(2)
COMB = bilan(S, ["dep", "commune", "nom", "codeEpci"]); COMB.to_csv(SORTIES / "commune_hectares_attendus_2027_INTERNE.csv", index=False)
EPCI = bilan(S, ["codeEpci"])
try:
    noms = pd.DataFrame(SESSION.get("https://geo.api.gouv.fr/epcis", params={"fields": "nom,code"}, timeout=120).json())
    EPCI = EPCI.merge(noms.rename(columns={"code": "codeEpci", "nom": "nom_epci"}), on="codeEpci", how="left")
except Exception as e:
    print("Noms d'interco indisponibles :", e)
EPCI.to_csv(SORTIES / "epci_hectares_attendus_2027.csv", index=False)
MAI = bilan(S, ["cx1", "cy1"]); MAI = MAI[MAI.n_pp >= MIN_PP_MAILLE]; MAI.to_csv(SORTIES / "maille1km_hectares_attendus_2027_INTERNE.csv", index=False)
print(f"Normandie : {S.p.mul(S.surface_ha).sum():,.0f} ha de prairies attendus convertis lors de la campagne 2027 "
      f"(probabilités calibrées ; intervalles élargis d'un facteur √{PHI:.1f})")
print(EPCI.sort_values("ha_attendus_campagne_2027", ascending=False).head(10)
      [[c for c in ["nom_epci", "ha_pp", "ha_attendus_campagne_2027", "ha_bas_90", "ha_haut_90", "taux_attendu"] if c in EPCI]].to_string(index=False))

In [ ]:
# F. Explicabilité : importance des variables et courbes de dépendance
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
ech = base[base.a0 <= 2022]; ech = ech.sample(min(len(ech), 60000), random_state=0)
imp = permutation_importance(m, ech[cols], ech["y2"], scoring="average_precision", n_repeats=3, random_state=0)
IMP = pd.Series(imp.importances_mean, index=cols).sort_values(ascending=False)
print(IMP.round(4).to_string()); IMP.to_csv(SORTIES / "importance_variables.csv")
# Courbes : seulement les variables continues (le calcul échoue sur les variables 0/1), méthode « brute », 5 000 prairies
continues = [c for c in IMP.index if ech[c].nunique() > 10][:4]
petit = ech[cols].astype(float).sample(min(len(ech), 5000), random_state=0)
fig, ax = plt.subplots(1, len(continues), figsize=(4 * len(continues), 3.5))
PartialDependenceDisplay.from_estimator(m, petit, continues, ax=ax, kind="average", method="brute", grid_resolution=20)
plt.tight_layout(); plt.savefig(FIG / "dependances_partielles.png", dpi=150); plt.show()
if MODELE_RETENU == "régression logistique":
    coef = pd.Series(m[-1].coef_[0], index=cols).sort_values()
    print("\nCoefficients standardisés (sens de l'effet) :"); print(coef.round(3).to_string())

In [ ]:
# G. Sobriété numérique + dictionnaire des données + résumé
duree = (time.time() - T_DEBUT) / 60
kg, kwh = float("nan"), float("nan")
if TRACKER is not None:
    kg = TRACKER.stop() or float("nan")
    e = getattr(TRACKER, "final_emissions_data", None)
    kwh = getattr(e, "energy_consumed", float("nan")) if e else float("nan")
print(f"Ce notebook : {duree:.0f} min, {kwh:.3f} kWh, {kg * 1000:.1f} g CO2e (estimation CodeCarbon)")
# Enregistré dans un fichier, avec son périmètre exact (notebook 3 seul, téléchargements des notebooks 1-2 non inclus)
pd.DataFrame([{"perimetre": "notebook 3 seul, données RPG déjà en cache (téléchargements des notebooks 1 et 2 non inclus)",
               "duree_min": round(duree, 1), "energie_kwh": round(kwh, 4), "emissions_g_co2e": round(kg * 1000, 2),
               "outil": "CodeCarbon (estimation)"}]).to_csv(SORTIES / "sobriete_v5.csv", index=False)
DICO = '''# Dictionnaire des données – Prairie Vigie
Sources : RPG parcelles graphiques 2015-2024 et grandes catégories 2025 (IGN, Licence Ouverte) ; communes et intercos (API Géo, Etalab) ;
zones humides (DREAL Normandie), aires d'alimentation de captages (Sandre), habitats Natura 2000 de Normandie occidentale, haies (BD TOPO, IGN).

Définitions
- Prairie permanente : code_group 18 du RPG.
- Conversion présumée : prairie permanente dont plus de 50 % de la surface est déclarée en terre arable l'année suivante
  (déclarations PAC, sans constat de terrain). Les colonnes n_retournees, ha_retournees et ha_retournes gardent leur nom
  historique : elles comptent des conversions présumées.
- Création : prairie permanente dont plus de 50 % de la surface était déclarée en terre arable l'année précédente.
- Sortie du RPG : prairie permanente absente des déclarations l'année suivante (n_sorties_rpg = nombre de prairies, pas des hectares).
- Prairie ancienne : jamais cultivée depuis 2015.
Aucune donnée personnelle ; résultats masqués sous 5 prairies. Prédictions diffusées seulement par interco et maille de 10 km.

Fichiers diffusables
- surfaces_par_dep_annee_2015_2025.csv : surfaces par classe (pp, herbe, arable, gel, autre) en ha, par département et année
- bilan_transitions_2019_2025.csv : par département et transition, n_pp, ha_pp, n_retournees, ha_retournees, n_sorties_rpg, n_gains, ha_gains
- observatoire_communes_2019_2025.csv : ha_retournes (conversions présumées) et ha_crees par commune et transition
- qualite_etiquettes.csv : persistance à 2 ans des conversions, sorties du RPG et conversions différées
- cultures_de_remplacement.csv : cultures déclarées l'année qui suit une conversion
- concordance_retournement_declare_2024.csv : comparaison au champ déclaratif retournmt_ du RPG 2024 (exploratoire ; champ toujours à 0 dans nos données)
- resultats_modeles_normandie.csv, verification_prediction_2025.csv, evaluation_complete.csv : performances (lift, rappel, AUC-PR)
- effet_regime_2023_2024.csv : hectares observés vs attendus, campagne 2022→23 (témoin) et 2023→24
- epci_hectares_attendus_2027.csv : hectares attendus convertis lors de la campagne 2027 par interco (intervalle à 90 %)
- importance_variables.csv, figures/*.png : explicabilité et calibration
- sobriete_v5.csv : durée et énergie du notebook 3 ; versions_bibliotheques.txt : versions utilisées
- Notebook 4 : enjeux_communes_2025.csv (faits mesurés, sans prédiction), priorites_epci_2027.csv, priorites_maille10km_2027.csv,
  priorites_synthese_normandie.csv, valeur_ajoutee_modele_vs_historique.csv (test 5 bis), valeur_ajoutee_parcelle.csv (test 5 ter),
  communes_bilan_prairies_2019_2025.csv, communes/intercos/mailles10km.geojson (carte)

Fichiers internes, jamais diffusés : caches, tous les .parquet (dont *_PRIVE), fichiers *_INTERNE.csv.
'''
(SORTIES / "DICTIONNAIRE_DES_DONNEES.md").write_text(DICO, encoding="utf-8")
print("\nRésumé des règles :")
print("  A1 persistance ≥ 75 % :", "OK" if CRIT_PERSIST else "ÉCHEC")
print("  B modèle retenu :", MODELE_RETENU)
print("\nRappel : ne jamais diffuser les caches, les .parquet (dont *_PRIVE) ni les fichiers *_INTERNE.csv.")